# 02 - Single-Qubit Gates

Every quantum computation is built from **gates** -- unitary transformations that rotate the state of a qubit on the Bloch sphere. In this notebook we explore the standard single-qubit gates available in Goqu:

- **Pauli gates** (X, Y, Z) -- 180-degree rotations around the three Bloch sphere axes
- **Hadamard** (H) -- creates equal superposition
- **Phase gates** (S, T) -- fractional Z-rotations forming the S^2 = Z, T^2 = S, T^4 = Z chain
- **Rotation gates** (RX, RY, RZ) -- continuous rotations, universal for single-qubit operations
- **Gate inverses** and **gate powers**

Geometrically, every single-qubit gate is a rotation of the Bloch sphere. A gate's 2x2 unitary matrix encodes the axis and angle of that rotation. We will visualize these rotations using Goqu's Bloch sphere renderer.

By the end of this notebook you will be able to:

1. **Describe** the Pauli, phase, and rotation gates and their matrix representations.
2. **Implement** any single-qubit rotation using RX, RY, RZ gates.
3. **Verify** the S²=Z, T²=S, T⁴=Z hierarchy using gate power operations.
4. **Explain** why any single-qubit unitary can be decomposed into three rotations.

In [ ]:
import (
	"fmt"
	"math"
	"math/cmplx"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/gate"
	"github.com/splch/goqu/sim/statevector"
	"github.com/splch/goqu/viz"
)

## Pauli Gates: X, Y, Z

The three Pauli gates are 180-degree rotations around the X, Y, and Z axes of the Bloch sphere:

| Gate | Action | Matrix |
|------|--------|--------|
| **X** (bit-flip) | \|0> -> \|1>, \|1> -> \|0> | `[[0,1],[1,0]]` |
| **Y** | \|0> -> i\|1>, \|1> -> -i\|0> | `[[0,-i],[i,0]]` |
| **Z** (phase-flip) | \|0> -> \|0>, \|1> -> -\|1> | `[[1,0],[0,-1]]` |

All three are **self-inverse**: applying the same gate twice returns the qubit to its original state.

In [ ]:
%%
// Apply X gate to |0> -- should produce |1>
c, _ := builder.New("X-gate", 1).X(0).Build()
sim := statevector.New(1)
sim.Evolve(c)
sv := sim.StateVector()
fmt.Println("X|0> =", sv)
fmt.Println("Circuit:")
fmt.Println(draw.String(c))

// Print the Pauli gate matrices
for _, info := range []struct {
	name string
	g    gate.Gate
}{
	{"X", gate.X},
	{"Y", gate.Y},
	{"Z", gate.Z},
} {
	m := info.g.Matrix()
	fmt.Printf("%s matrix: [[%v, %v], [%v, %v]]\n", info.name, m[0], m[1], m[2], m[3])
}

## The Hadamard Gate

The Hadamard gate **H** is arguably the most important single-qubit gate. It maps computational basis states to superposition states:

- H|0> = |+> = (|0> + |1>) / sqrt(2)
- H|1> = |-> = (|0> - |1>) / sqrt(2)

On the Bloch sphere, H is a 180-degree rotation around the axis halfway between X and Z. Like the Pauli gates, H is self-inverse: H^2 = I.

In [ ]:
%%
// Apply H to |0> and visualize on the Bloch sphere
c, _ := builder.New("H-gate", 1).H(0).Build()
sim := statevector.New(1)
sim.Evolve(c)
sv := sim.StateVector()
fmt.Println("H|0> =", sv)
fmt.Printf("  |0> amplitude: %.4f\n", sv[0])
fmt.Printf("  |1> amplitude: %.4f\n", sv[1])
fmt.Printf("  Both equal 1/sqrt(2) = %.4f\n", 1/math.Sqrt(2))
fmt.Println()
fmt.Println("Circuit:")
fmt.Println(draw.String(c))

// Render on the Bloch sphere -- |+> points along the +X axis
gonbui.DisplayHTML(viz.Bloch(sv, viz.WithTitle("|+> = H|0>")))

## Phase Gates: S and T

Phase gates rotate the qubit around the Z axis by fractional angles:

| Gate | Phase on |1> | Relation |
|------|------------|----------|
| **Z** | e^(i*pi) = -1 | pi rotation |
| **S** | e^(i*pi/2) = i | S^2 = Z |
| **T** | e^(i*pi/4) | T^2 = S, T^4 = Z |

The T gate is critical for universal quantum computing: the **Clifford+T** gate set can approximate any unitary to arbitrary precision (Solovay-Kitaev theorem).

Let's verify the S^2 = Z and T^2 = S chain by comparing gate matrices.

In [ ]:
%%
// Helper to compare two gate matrices
matricesEqual := func(a, b []complex128, tol float64) bool {
	if len(a) != len(b) {
		return false
	}
	for i := range a {
		if cmplx.Abs(a[i]-b[i]) > tol {
			return false
		}
	}
	return true
}

// S^2 = Z
s2 := gate.Pow(gate.S, 2)
fmt.Println("S^2 matrix:", s2.Matrix())
fmt.Println("Z   matrix:", gate.Z.Matrix())
fmt.Println("S^2 == Z?", matricesEqual(s2.Matrix(), gate.Z.Matrix(), 1e-10))
fmt.Println()

// T^2 = S
t2 := gate.Pow(gate.T, 2)
fmt.Println("T^2 matrix:", t2.Matrix())
fmt.Println("S   matrix:", gate.S.Matrix())
fmt.Println("T^2 == S?", matricesEqual(t2.Matrix(), gate.S.Matrix(), 1e-10))
fmt.Println()

// T^4 = Z
t4 := gate.Pow(gate.T, 4)
fmt.Println("T^4 matrix:", t4.Matrix())
fmt.Println("Z   matrix:", gate.Z.Matrix())
fmt.Println("T^4 == Z?", matricesEqual(t4.Matrix(), gate.Z.Matrix(), 1e-10))

## Rotation Gates: RX, RY, RZ

Rotation gates provide **continuous** rotation around the Bloch sphere axes:

- **RX(theta)** = exp(-i * theta/2 * X) -- rotation around the X axis
- **RY(theta)** = exp(-i * theta/2 * Y) -- rotation around the Y axis  
- **RZ(theta)** = exp(-i * theta/2 * Z) -- rotation around the Z axis

These gates are **universal**: any single-qubit unitary can be decomposed as a sequence of rotations RZ(a) * RY(b) * RZ(c) (the ZYZ Euler decomposition).

Let's sweep RX from 0 to 2*pi and observe how the statevector evolves.

In [ ]:
%%
// RX(theta) sweep from 0 to 2*pi
fmt.Println("RX(theta)|0> at several angles:")
fmt.Println("------------------------------------------")
angles := []struct {
	name  string
	theta float64
}{
	{"0", 0},
	{"pi/4", math.Pi / 4},
	{"pi/2", math.Pi / 2},
	{"pi", math.Pi},
	{"3pi/2", 3 * math.Pi / 2},
	{"2pi", 2 * math.Pi},
}

for _, a := range angles {
	c, _ := builder.New("rx", 1).RX(a.theta, 0).Build()
	sim := statevector.New(1)
	sim.Evolve(c)
	sv := sim.StateVector()
	p0 := real(sv[0])*real(sv[0]) + imag(sv[0])*imag(sv[0])
	p1 := real(sv[1])*real(sv[1]) + imag(sv[1])*imag(sv[1])
	fmt.Printf("  RX(%6s)|0> = [%8.4f, %8.4f]  P(|0>)=%.3f  P(|1>)=%.3f\n",
		a.name, sv[0], sv[1], p0, p1)
}

// Visualize RX(pi/2)|0> on the Bloch sphere
c, _ := builder.New("rx-pi/2", 1).RX(math.Pi/2, 0).Build()
sim := statevector.New(1)
sim.Evolve(c)
gonbui.DisplayHTML(viz.Bloch(sim.StateVector(), viz.WithTitle("RX(pi/2)|0>")))

## Gate Inverses

Every quantum gate is unitary, which means it has an inverse (its conjugate transpose, or **adjoint**). Applying a gate followed by its inverse returns the qubit to its original state.

In Goqu, call `.Inverse()` on any gate to get its adjoint:
- `gate.H.Inverse()` returns H (since H is self-inverse)
- `gate.S.Inverse()` returns S-dagger (Sdg)
- `gate.T.Inverse()` returns T-dagger (Tdg)

In [ ]:
%%
// Apply a gate then its inverse -- should return to |0>
fmt.Println("Gate -> Inverse -> back to |0>:")
fmt.Println("-------------------------------")

for _, info := range []struct {
	name string
	g    gate.Gate
}{
	{"H", gate.H},
	{"S", gate.S},
	{"T", gate.T},
	{"X", gate.X},
	{"RX(pi/3)", gate.RX(math.Pi / 3)},
} {
	inv := info.g.Inverse()
	c, _ := builder.New("inv", 1).
		Apply(info.g, 0).
		Apply(inv, 0).
		Build()
	sim := statevector.New(1)
	sim.Evolve(c)
	sv := sim.StateVector()
	p0 := real(sv[0])*real(sv[0]) + imag(sv[0])*imag(sv[0])
	fmt.Printf("  %10s then %s^dag : P(|0>) = %.6f\n", info.name, info.name, p0)
}

// Show that S.Inverse() == Sdg
fmt.Println()
fmt.Println("S.Inverse() name:", gate.S.Inverse().Name())
fmt.Println("T.Inverse() name:", gate.T.Inverse().Name())

## Predict-Then-Verify: What state does RY(pi/2)|0> produce?

**Think before running the next cell.** The gate RY(pi/2) rotates |0> (the north pole of the Bloch sphere) by 90 degrees around the Y axis. Where does this land?

Hint: A 90-degree Y-rotation from the north pole lands on the +X axis, which is the |+> state. This is the same state that H|0> produces!

In [ ]:
%%
// RY(pi/2)|0> should produce |+> (same as H|0>)
cRY, _ := builder.New("ry", 1).RY(math.Pi/2, 0).Build()
cH, _ := builder.New("h", 1).H(0).Build()

simRY := statevector.New(1)
simRY.Evolve(cRY)
svRY := simRY.StateVector()

simH := statevector.New(1)
simH.Evolve(cH)
svH := simH.StateVector()

fmt.Println("RY(pi/2)|0> =", svRY)
fmt.Println("H|0>        =", svH)

// They both have equal |0> and |1> amplitudes (both real, both 1/sqrt(2))
fmt.Printf("\nBoth produce equal superposition with P(|0>) = P(|1>) = 0.5\n")
fmt.Printf("RY version: P(|0>)=%.4f, P(|1>)=%.4f\n",
	real(svRY[0])*real(svRY[0])+imag(svRY[0])*imag(svRY[0]),
	real(svRY[1])*real(svRY[1])+imag(svRY[1])*imag(svRY[1]))
fmt.Printf("H  version: P(|0>)=%.4f, P(|1>)=%.4f\n",
	real(svH[0])*real(svH[0])+imag(svH[0])*imag(svH[0]),
	real(svH[1])*real(svH[1])+imag(svH[1])*imag(svH[1]))

// Visualize both side by side
gonbui.DisplayHTML(
	"<div style='display:flex;gap:20px'>" +
		viz.Bloch(svRY, viz.WithTitle("RY(pi/2)|0>"), viz.WithSize(300, 300)) +
		viz.Bloch(svH, viz.WithTitle("H|0>"), viz.WithSize(300, 300)) +
		"</div>")

## Power of Gates

Goqu provides `gate.Pow(g, k)` to raise a gate to an integer power by repeated matrix multiplication. This lets us verify algebraic identities:

- T^2 = S
- S^2 = Z
- T^4 = Z
- X^2 = I (self-inverse)
- H^2 = I (self-inverse)

In [ ]:
%%
// Helper to compare matrices
matEq := func(a, b []complex128) bool {
	for i := range a {
		if cmplx.Abs(a[i]-b[i]) > 1e-10 {
			return false
		}
	}
	return true
}

// gate.Pow(T, 2) == S
fmt.Println("gate.Pow(T, 2) == S?", matEq(gate.Pow(gate.T, 2).Matrix(), gate.S.Matrix()))

// gate.Pow(S, 2) == Z
fmt.Println("gate.Pow(S, 2) == Z?", matEq(gate.Pow(gate.S, 2).Matrix(), gate.Z.Matrix()))

// gate.Pow(T, 4) == Z
fmt.Println("gate.Pow(T, 4) == Z?", matEq(gate.Pow(gate.T, 4).Matrix(), gate.Z.Matrix()))

// gate.Pow(X, 2) == I
fmt.Println("gate.Pow(X, 2) == I?", matEq(gate.Pow(gate.X, 2).Matrix(), gate.I.Matrix()))

// gate.Pow(H, 2) == I
fmt.Println("gate.Pow(H, 2) == I?", matEq(gate.Pow(gate.H, 2).Matrix(), gate.I.Matrix()))

## Exercises

### Exercise 1: Build H from RY and RZ

The Hadamard gate can be decomposed using rotation gates. One decomposition (up to global phase) is:

**H = RZ(pi) * RY(pi/2)**

Build a circuit that applies RY(pi/2) then RZ(pi) and verify that the resulting state matches H|0>. Note that the two circuits may differ by a global phase, but the measurement probabilities will be identical.

In [ ]:
%%
// Exercise 1: Build H from RY and RZ
// Decomposition (up to global phase): H = RZ(pi) * RY(pi/2)
// In circuit order: first RY, then RZ
//
// TODO: Fill in the rotation angles to reproduce H|0>
// cDecomp, _ := builder.New("h-decomp", 1).
// 	RY( /* angle */ , 0).
// 	RZ( /* angle */ , 0).
// 	Build()
//
// cH, _ := builder.New("h-direct", 1).H(0).Build()
//
// simD := statevector.New(1)
// simD.Evolve(cDecomp)
// svD := simD.StateVector()
//
// simH := statevector.New(1)
// simH.Evolve(cH)
// svH := simH.StateVector()
//
// fmt.Println("RZ*RY|0> =", svD)
// fmt.Println("H|0>     =", svH)
//
// // Check measurement probabilities match
// p0D := real(svD[0])*real(svD[0]) + imag(svD[0])*imag(svD[0])
// p1D := real(svD[1])*real(svD[1]) + imag(svD[1])*imag(svD[1])
// p0H := real(svH[0])*real(svH[0]) + imag(svH[0])*imag(svH[0])
// p1H := real(svH[1])*real(svH[1]) + imag(svH[1])*imag(svH[1])
// fmt.Printf("\nDecomposed: P(|0>)=%.4f, P(|1>)=%.4f\n", p0D, p1D)
// fmt.Printf("Hadamard:   P(|0>)=%.4f, P(|1>)=%.4f\n", p0H, p1H)
// fmt.Println("Probabilities match:", math.Abs(p0D-p0H) < 1e-10 && math.Abs(p1D-p1H) < 1e-10)
// fmt.Println("\nDecomposed circuit:")
// fmt.Println(draw.String(cDecomp))
fmt.Println("Uncomment the code above and fill in the rotation angles!")

### Exercise 2: U3 is Universal

The `gate.U3(theta, phi, lambda)` gate can produce **any** single-qubit state from |0>. Verify this by:
1. Targeting a specific state on the Bloch sphere (e.g., the -Y axis: |0> - i|1>)/sqrt(2))
2. Finding the right U3 parameters to reach it
3. Confirming with the Bloch sphere visualization

In [ ]:
%%
// Exercise 2: Use U3 to produce various states on the Bloch sphere
// U3(theta, phi, lambda) is the most general single-qubit gate.
//
// TODO: Find the U3 parameters that produce each target state:
//   State 1: |1>   (south pole)
//   State 2: |+>   (equator, +X axis)
//   State 3: |-i>  = (|0> - i|1>)/sqrt(2)  (equator, -Y axis)
//   State 4: An arbitrary point of your choice
//
// Hint: U3(theta, phi, lambda)|0> = cos(theta/2)|0> + e^(i*phi)*sin(theta/2)|1>
//
// c1, _ := builder.New("u3-ket1", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim1 := statevector.New(1)
// sim1.Evolve(c1)
// sv1 := sim1.StateVector()
// fmt.Println("U3(...)|0> =", sv1, "  (should be |1>)")
//
// c2, _ := builder.New("u3-plus", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim2 := statevector.New(1)
// sim2.Evolve(c2)
// sv2 := sim2.StateVector()
// fmt.Println("U3(...)|0> =", sv2, "  (should be |+>)")
//
// c3, _ := builder.New("u3-mi", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim3 := statevector.New(1)
// sim3.Evolve(c3)
// sv3 := sim3.StateVector()
// fmt.Println("U3(...)|0> =", sv3, "  (should be |-i>)")
//
// c4, _ := builder.New("u3-arb", 1).U3( /* theta, phi, lambda */ , 0).Build()
// sim4 := statevector.New(1)
// sim4.Evolve(c4)
// sv4 := sim4.StateVector()
// fmt.Printf("U3(...)|0> = [%.4f, %.4f]\n", sv4[0], sv4[1])
//
// gonbui.DisplayHTML(
// 	"<div style='display:flex;flex-wrap:wrap;gap:10px'>" +
// 		viz.Bloch(sv1, viz.WithTitle("|1>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv2, viz.WithTitle("|+>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv3, viz.WithTitle("|-i>"), viz.WithSize(280, 280)) +
// 		viz.Bloch(sv4, viz.WithTitle("Arbitrary"), viz.WithSize(280, 280)) +
// 		"</div>")
fmt.Println("Uncomment the code above and fill in the U3 parameters!")

## Key Takeaways

1. **Every single-qubit gate is a rotation** of the Bloch sphere, fully described by a 2x2 unitary matrix.

2. **Pauli gates** (X, Y, Z) are 180-degree rotations and are self-inverse.

3. **Hadamard** creates equal superposition and is the gateway to quantum parallelism.

4. **Phase gates** form a hierarchy: T^2 = S, S^2 = Z. The T gate is essential for universality.

5. **Rotation gates** (RX, RY, RZ) provide continuous rotations and are universal: any single-qubit unitary = RZ * RY * RZ (Euler decomposition).

6. **U3(theta, phi, lambda)** is the most general single-qubit gate -- it can reach any point on the Bloch sphere from |0>.

7. **Inverses** undo gates (gate followed by its inverse = identity). Use `g.Inverse()` in Goqu.

8. **Powers** let you compose a gate with itself: `gate.Pow(g, k)` multiplies the matrix k times.

In the next notebook, we will combine single-qubit gates with multi-qubit gates to create **entanglement** -- correlations that have no classical analog.